# 01 · Comparing decision criteria

Exploratory walkthrough of the budget allocation models on the synthetic sample dataset. See `docs/model.md` for the formulation.

In [1]:
import pandas as pd
from budget_opt import load_problem, solve_scenario, solve_expected, solve_robust

problem = load_problem('../data/sample/budget_items.csv', '../data/sample/departments.csv', total_budget=1000)
problem.items.head()

,item_id,item_name,department,min_amount,max_amount,return_recession,return_stable,return_growth,mandatory
0,I01,Raw materials,Production,48.7,48.7,1.182,1.182,1.182,True
1,I02,Equipment maintenance,Production,44.9,74.3,1.479,1.343,1.207,False
2,I03,Energy,Production,49.3,49.3,1.167,1.088,1.009,True
3,I04,Quality control,Production,28.5,141.2,1.421,1.243,1.065,False
4,I05,Digital advertising,Sales & Marketing,32.2,74.9,1.067,1.216,1.365,False


## Effect of each plan under every scenario

Rows are plans (criteria), columns are the scenario that actually happens. The robust plan has the highest worst case and equalises the effect across scenarios.

In [2]:
plans = {
    'recession': solve_scenario(problem, 'recession'),
    'stable': solve_scenario(problem, 'stable'),
    'growth': solve_scenario(problem, 'growth'),
    'expected': solve_expected(problem),
    'robust': solve_robust(problem),
}
payoff = pd.DataFrame({k: v.effects for k, v in plans.items()}).T.round(1)
payoff['worst_case'] = payoff.min(axis=1)
payoff

,recession,stable,growth,worst_case
recession,1196.8,1191.1,1185.5,1185.5
stable,1170.8,1201.5,1232.2,1170.8
growth,1160.4,1200.9,1241.3,1160.4
expected,1170.8,1201.5,1232.2,1170.8
robust,1194.1,1194.1,1194.1,1194.1


## Regret matrix (Savage criterion)

Regret = best achievable effect in a scenario minus the effect of the plan.

In [3]:
regret = (payoff.drop(columns='worst_case').max() - payoff.drop(columns='worst_case')).round(1)
regret['max_regret'] = regret.max(axis=1)
regret.sort_values('max_regret')

,recession,stable,growth,max_regret
stable,26.0,0.0,9.1,26.0
expected,26.0,0.0,9.1,26.0
growth,36.4,0.6,0.0,36.4
robust,2.7,7.4,47.2,47.2
recession,0.0,10.4,55.8,55.8


## Where the robust plan differs from the expected-value plan

In [4]:
diff = plans['expected'].to_frame(problem)[['item_name', 'department']].copy()
diff['expected'] = plans['expected'].allocation.values.round(1)
diff['robust'] = plans['robust'].allocation.values.round(1)
diff['delta'] = (diff['robust'] - diff['expected']).round(1)
diff[diff['delta'].abs() > 0.05].sort_values('delta')

,item_name,department,expected,robust,delta
6,Sales bonuses,Sales & Marketing,133.1,74.4,-58.7
16,Warehousing,Logistics,63.6,33.6,-30.0
9,Process automation,R&D,33.5,58.7,25.2
3,Quality control,Production,77.7,141.2,63.5
